## 1. Unified Setup and Paths

In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from pathlib import Path
import joblib  
import torch

# --- CONSTANTS & PATHS ---
WINDOW_SIZE = 30
MAX_RUL = 125
TARGET = "RUL_clipped"

# 1. Load Data
DATA_PATH = Path("../data/experiments/DS-C")
MODEL_PATH_ML = Path("../artifacts/random_forest_model.joblib")
MODEL_PATH_DL = Path("../artifacts/lstm_model.pth")

train_df = pd.read_csv(DATA_PATH / "train.csv")
test_df = pd.read_csv(DATA_PATH / "test.csv")

# Identify PC features (PC_1 to PC_33)
feature_cols = [c for c in train_df.columns if c.startswith('PC_')]



In [ ]:
import torch.nn as nn
from sklearn.ensemble import RandomForestRegressor

# --- 1. Model Definitions & Scoring ---

# NASA Scoring Function (From your DS-C notebook)
def nasa_scoring_function(y_true, y_pred):
    d = y_pred - y_true
    score = np.where(d < 0, np.exp(-d/13)-1, np.exp(d/10)-1)
    return np.sum(score)

# LSTM Class definition (Required to load the .pth model)
class LSTMModel(nn.Module):
    def __init__(self, input_dim=17, hidden_dim=128, num_layers=2, output_dim=1):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # The LSTM layer
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        
        # The final layer must be named 'head' to match your saved state_dict
        # The keys 'head.0' and 'head.3' suggest a Sequential block like this:
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 64), # head.0
            nn.ReLU(),                 # head.1
            nn.Dropout(0.2),           # head.2
            nn.Linear(64, output_dim)  # head.3
        )

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        # Pass the last time step through the 'head'
        out = self.head(out[:, -1, :]) 
        return out

# --- 2. Load the Models ---

# Load Traditional ML (Random Forest)
# Assuming 'rf_model' is your best model from DS-C
rf_model = joblib.load(MODEL_PATH_ML)

# Load Deep Learning (LSTM)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_dl = LSTMModel(input_dim=17, hidden_dim=128, num_layers=2).to(device)
model_dl.load_state_dict(torch.load(MODEL_PATH_DL, map_location=device))
model_dl.eval()
# --- 3. Differentiated Data Preparation ---

# Random Forest uses all 33 PCs
ml_feature_cols = [f'PC_{i}' for i in range(1, 34)]

# LSTM only uses the first 17 PCs (to match your 17 input_dim checkpoint)
dl_feature_cols = [f'PC_{i}' for i in range(1, 18)] 

target_col = 'RUL_clipped' 

# A. Prepare for Random Forest (All 33 Features)
test_ml = test_df.groupby('unit_number').last().reset_index()
X_ml = test_ml[ml_feature_cols]
y_true = test_ml[target_col]
unit_ids = test_ml['unit_number']

# B. Prepare for LSTM (Only 17 Features)
dl_windows = []
for unit in unit_ids:
    unit_data = test_df[test_df['unit_number'] == unit]
    # Filter for ONLY the 17 features the LSTM expects
    window = unit_data[dl_feature_cols].values[-WINDOW_SIZE:]
    dl_windows.append(window)

# Convert to tensor - This will now be [Batch, 30, 17]
X_dl_tensor = torch.tensor(np.array(dl_windows), dtype=torch.float32).to(device)

# --- 4. Get Predictions & Create Summary ---

# Predictions
preds_rf = rf_model.predict(X_ml)

model_dl.eval()
with torch.no_grad():
    # This will now work because dimensions match (17 features)
    preds_dl = model_dl(X_dl_tensor).cpu().numpy().flatten()

# Summary DataFrame
summary_df = pd.DataFrame({
    'Unit': unit_ids.astype(int),
    'Actual_RUL': y_true.values,
    'RF_Prediction': preds_rf,
    'LSTM_Prediction': preds_dl
})

# Give 70% weight to the better performing model
summary_df['Ensemble_Prediction'] = (summary_df['RF_Prediction'] * 0.90) + (summary_df['LSTM_Prediction'] * 0.10)

# Final Scoring
ensemble_nasa = nasa_scoring_function(summary_df['Actual_RUL'], summary_df['Ensemble_Prediction'])

print(f"Ensemble NASA Score: {ensemble_nasa:.2f}")
display(summary_df.head(10))

Ensemble NASA Score: 739.34


,Unit,Actual_RUL,RF_Prediction,LSTM_Prediction,Ensemble_Prediction
0,1,112.0,114.684132,113.764412,114.684132
1,2,98.0,111.315674,115.637474,111.315674
2,3,69.0,57.159474,109.643730,57.159474
3,4,82.0,97.239630,117.699944,97.239630
4,5,91.0,107.154252,116.772903,107.154252
5,6,93.0,108.412199,113.632523,108.412199
6,7,91.0,100.122445,119.473656,100.122445
7,8,95.0,102.966371,113.907784,102.966371
8,9,111.0,109.960406,115.224548,109.960406
9,10,96.0,106.412000,116.667702,106.412000
